### Vector stores and retrievers

This video tutorial will familiarize you with LangChain's vector store and retriever abstractions. These abstractions are designed to support retrieval of data -- from (vector) databases and other sources -- for integration with LI.M workflows. They are importa for applications that fetch data to be reasoned over as part of model inference, as in the case of retrieval-augmented generation.

We will cover

- Documents
- Vector stores
- Retrievers

### Documents

LangChain implements a Document abstraction, which is intended to represent a nit of text and associated
metadata. It has two attributes:

- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata. The metadata attribute can capture information about the
source of the document, its relationship to other documents, and other information. Note that an
individual Document object often represents a chunk of a larger document.

Let's generate some sample documents:

In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
]

In [2]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.')]

In [23]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")

os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

llm=ChatGroq(groq_api_key=groq_api_key, model="llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000294DF201EB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000294DF203BF0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore=Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)
vectorstore

In [10]:
vectorstore.similarity_search("What is cat?")

[Document(id='457378bd-936b-48fa-8807-e3b34ca3484d', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='39e3a2d6-6fc5-4287-9845-927d42ff0da0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='7e5fed0b-62ee-4978-83b3-245425fdefb0', metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.')]

In [11]:
await vectorstore.asimilarity_search_with_score("What is cat?")

[(Document(id='457378bd-936b-48fa-8807-e3b34ca3484d', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.8121948838233948),
 (Document(id='39e3a2d6-6fc5-4287-9845-927d42ff0da0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5733122825622559),
 (Document(id='7e5fed0b-62ee-4978-83b3-245425fdefb0', metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
  1.7853065729141235)]

### Retrievers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated
into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous
and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what
method we wish to use to retrieve documents, we can create a runnable easily. Below we will build
one around the similarity search method:

In [14]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat", "dog"])

[[Document(id='457378bd-936b-48fa-8807-e3b34ca3484d', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='39e3a2d6-6fc5-4287-9845-927d42ff0da0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a
VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [16]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}
)
retriever.batch(["cat", "dog"])

[[Document(id='457378bd-936b-48fa-8807-e3b34ca3484d', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='39e3a2d6-6fc5-4287-9845-927d42ff0da0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [29]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.
{question}

Context :
{context}

"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

chain={"context":retriever,"question": RunnablePassthrough()}|prompt|llm

response=chain.invoke("cats or dogs? based on loyality")
response.content

'Dogs. The document states that dogs are known for their loyalty.'